# 06 — Map Layers Prototype

**Story**: S01-06 | **Spec**: `openspec/capabilities/map/spec.md`

Validate all map tile sources and WMS layers for the BoatOps planning dashboard.
No credentials required — all sources are public.

| Scenario | Validates |
|----------|-----------|
| SCENARIO-MAP-OSM-001 | OSM tiles at zoom 10 near Warnemünde |
| SCENARIO-MAP-SEAMAP-001 | OpenSeaMap seamark overlay |
| SCENARIO-MAP-GEOTIFF-001 | Synthetic raster as georeferenced ImageOverlay |
| SCENARIO-MAP-WMS-001 | Generic WMS layer loading (validated via GEBCO) |
| SCENARIO-MAP-GEBCO-001 | GEBCO bathymetry WMS (`GEBCO_LATEST`) |
| SCENARIO-MAP-EMODNET-001 | EMODnet Seabed Habitats WMS (`eusm2022`) |

**AOI center**: Warnemünde 54.18°N, 12.08°E | **AOI bbox**: `[9.5, 53.5, 15.0, 56.0]`

In [ ]:
import io
import json
import base64
import warnings
import xml.etree.ElementTree as ET
from math import radians, floor, log, tan, cos, pi

import numpy as np
import requests
import matplotlib.pyplot as plt
import cmocean
import folium

from PIL import Image

from ipyleaflet import (
    Map, TileLayer, WMSLayer, ImageOverlay,
    LayersControl, ScaleControl, FullScreenControl,
)

In [ ]:
AOI_CENTER = (54.18, 12.08)
AOI_ZOOM   = 8
AOI_BBOX   = (9.5, 53.5, 15.0, 56.0)   # (west, south, east, north)

OSM_URL     = 'https://tile.openstreetmap.org/{z}/{x}/{y}.png'
SEAMAP_URL  = 'https://tiles.openseamap.org/seamark/{z}/{x}/{y}.png'
GEBCO_WMS   = 'https://wms.gebco.net/mapserv'                 # old gebco.net/.../mapserv returns 404
EMODNET_WMS = 'https://ows.emodnet-seabedhabitats.eu/geoserver/emodnet_view/wms'
GEBCO_LAYER        = 'GEBCO_LATEST'
EMODNET_LAYER      = 'eusm_helcom_group'
CMEMS_BAL_STATIC   = 'cmems_mod_bal_phy_anfc_static'          # deptho = marine bathymetry

WMS_TEST_BBOX = (10.5, 53.8, 13.0, 55.2)
TILE_HEADERS  = {'User-Agent': 'BoatOps/1.0'}


# ── slippy-map tile index ────────────────────────────────────────────────────
def deg2tile(lat: float, lon: float, zoom: int) -> tuple:
    lat_r = radians(lat); n = 2 ** zoom
    x = floor((lon + 180.0) / 360.0 * n)
    y = floor((1.0 - log(tan(lat_r) + 1.0 / cos(lat_r)) / pi) / 2.0 * n)
    return x, y


# ── WMS discovery ────────────────────────────────────────────────────────────
def wms_layer_names(base_url: str, version: str = '1.3.0') -> list:
    resp = requests.get(base_url,
        params={'SERVICE': 'WMS', 'REQUEST': 'GetCapabilities', 'VERSION': version},
        timeout=30)
    resp.raise_for_status()
    root = ET.fromstring(resp.content)
    return [el.text.strip() for el in root.iter()
            if el.tag.split('}')[-1] == 'Name' and el.text and el.text.strip()]


# ── 2-D array → transparent ImageOverlay data URI ───────────────────────────
def _to_overlay(arr_2d, lats, cmap, norm, alpha: float = 0.75) -> str:
    """NaN cells → transparent; north-up flip for ascending-lat CMEMS grids.
    cmap may be a string name or a matplotlib/cmocean colormap object.
    """
    cmap_obj = cmap if callable(cmap) else plt.get_cmap(cmap)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fill = float(getattr(norm, 'vmin', None) or 0.0)
        safe = np.where(np.isnan(arr_2d), fill, arr_2d)
        rgba = cmap_obj(norm(safe)).copy()
    rgba[np.isnan(arr_2d),  3] = 0.0
    rgba[~np.isnan(arr_2d), 3] = alpha
    img = Image.fromarray((rgba * 255).astype(np.uint8), 'RGBA')
    if lats[0] < lats[-1]:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    buf = io.BytesIO(); img.save(buf, format='PNG'); buf.seek(0)
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()


# ── subsampled quiver → transparent PNG URI ──────────────────────────────────
def _to_quiver_uri(lons, lats, U, V, cmap=None, speed_max: float = 0.5,
                   stride: int = None) -> tuple:
    cmap = cmap if cmap is not None else cmocean.cm.speed
    if stride is None:
        stride = max(1, min(U.shape) // 15)
    sl, ss = lons[::stride], lats[::stride]
    sU, sV = U[::stride, ::stride], V[::stride, ::stride]
    sSpd = np.sqrt(sU**2 + sV**2)
    LG, AG = np.meshgrid(sl, ss)
    fig = plt.figure(figsize=(8, 8*(lats.max()-lats.min())/(lons.max()-lons.min())))
    fig.patch.set_alpha(0)
    ax = fig.add_axes([0, 0, 1, 1]); ax.patch.set_alpha(0)
    ax.set_xlim(lons.min(), lons.max()); ax.set_ylim(lats.min(), lats.max())
    ax.quiver(LG, AG, sU, sV, sSpd, cmap=cmap, scale=4, width=0.005,
              headwidth=3, headlength=4, minlength=0, clim=(0, speed_max))
    ax.axis('off')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, transparent=True, bbox_inches=None)
    plt.close(fig); buf.seek(0)
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode(), sSpd


# ── folium map with Leaflet.velocity animated particles ─────────────────────
def _make_velocity_map(lons, lats, U, V, title: str, max_vel: float,
                        color_scale=None, center=None, zoom: int = 7):
    """Phase 1 proof-of-concept for windy.com-style particle animation.
    ipyleaflet 0.20 lacks VelocityLayer; Phase 3 will use Leaflet.velocity natively.
    NaN values replaced with 0 (no particle movement in masked areas).
    """
    cx = center or [float((lats.min()+lats.max())/2), float((lons.min()+lons.max())/2)]
    if lats[0] < lats[-1]:
        U_ns = np.flipud(np.where(np.isnan(U), 0., U))
        V_ns = np.flipud(np.where(np.isnan(V), 0., V))
        lns = lats[::-1]
    else:
        U_ns, V_ns, lns = np.where(np.isnan(U),0.,U), np.where(np.isnan(V),0.,V), lats
    hdr = {'parameterCategory': 1, 'refTime': '2000-01-01 00:00:00',
           'la1': float(lns[0]),    'lo1': float(lons[0]),
           'la2': float(lns[-1]),   'lo2': float(lons[-1]),
           'dx': float(abs(lons[1]-lons[0])), 'dy': float(abs(lats[1]-lats[0])),
           'nx': int(lons.size),    'ny': int(lns.size)}
    vel = [{**hdr,'parameterNumber':2,'data':U_ns.flatten().tolist()},
           {**hdr,'parameterNumber':3,'data':V_ns.flatten().tolist()}]
    cs = color_scale or ['#fff9f0','#f9c862','#f47c2f','#c0392b','#6c1a1a']
    m = folium.Map(location=cx, zoom_start=zoom, tiles='OpenStreetMap')
    mn = m.get_name()
    m.get_root().header.add_child(folium.Element(
        '<script src="https://cdn.jsdelivr.net/npm/leaflet-velocity@1.9.1/dist/leaflet-velocity.min.js"></script>\n'
        '<link href="https://cdn.jsdelivr.net/npm/leaflet-velocity@1.9.1/dist/leaflet-velocity.css" rel="stylesheet">'
    ))
    m.get_root().script.add_child(folium.Element(
        f'(function(){{var d={json.dumps(vel)};'
        f'setTimeout(function(){{L.velocityLayer({{displayValues:true,'
        f'displayOptions:{{velocityType:"{title}",displayPosition:"bottomleft",'
        f'displayEmptyString:"No data"}},data:d,maxVelocity:{max_vel},'
        f'velocityScale:0.005,colorScale:{json.dumps(cs)}}}).addTo({mn});}},400);}})();'
    ))
    return m


print('Constants and all helpers loaded.')
print(f'AOI center: {AOI_CENTER} | zoom: {AOI_ZOOM}')

## 1. OSM Base Layer — SCENARIO-MAP-OSM-001

Fetch a tile near Warnemünde at zoom 10 to confirm connectivity and image format.

In [ ]:
zoom = 10
tx, ty = deg2tile(*AOI_CENTER, zoom)
print(f'Warnemünde tile at zoom {zoom}: x={tx}, y={ty}')

resp_osm = requests.get(
    f'https://tile.openstreetmap.org/{zoom}/{tx}/{ty}.png',
    headers=TILE_HEADERS, timeout=15,
)
assert resp_osm.status_code == 200, f'OSM tile HTTP {resp_osm.status_code}'
assert 'image' in resp_osm.headers.get('Content-Type', ''), 'Expected image content'
print(f'OK: HTTP 200, {len(resp_osm.content):,} bytes, {resp_osm.headers["Content-Type"]}')

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(Image.open(io.BytesIO(resp_osm.content)))
ax.set_title(f'OSM tile z={zoom}/x={tx}/y={ty}')
ax.axis('off')
plt.tight_layout()
plt.show()

m_osm = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_osm.add(TileLayer(
    url=OSM_URL,
    attribution='© OpenStreetMap contributors',
    name='OpenStreetMap',
))
m_osm.add(ScaleControl(position='bottomleft'))
m_osm

## 2. OpenSeaMap Seamark Overlay — SCENARIO-MAP-SEAMAP-001

OpenSeaMap returns HTTP 200 with a transparent PNG even for tiles with no seamarks.

In [ ]:
resp_sea = requests.get(
    f'https://tiles.openseamap.org/seamark/{zoom}/{tx}/{ty}.png',
    headers=TILE_HEADERS, timeout=15,
)
assert resp_sea.status_code == 200, f'OpenSeaMap tile HTTP {resp_sea.status_code}'
assert 'image' in resp_sea.headers.get('Content-Type', ''), 'Expected image'
print(f'OpenSeaMap: HTTP 200, {len(resp_sea.content):,} bytes, {resp_sea.headers.get("Content-Type")}')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(Image.open(io.BytesIO(resp_osm.content)))
axes[0].set_title('OSM base')
axes[0].axis('off')
axes[1].imshow(Image.open(io.BytesIO(resp_sea.content)))
axes[1].set_title('OpenSeaMap (transparent PNG)')
axes[1].axis('off')
plt.suptitle(f'z={zoom}/x={tx}/y={ty} near Warnemünde')
plt.tight_layout()
plt.show()

m_seamap = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_seamap.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m_seamap.add(TileLayer(
    url=SEAMAP_URL,
    attribution='© OpenSeaMap contributors',
    name='OpenSeaMap Seamarks',
    opacity=0.9,
))
m_seamap.add(LayersControl(position='topright'))
m_seamap.add(ScaleControl(position='bottomleft'))
m_seamap

## 3. GeoTIFF Overlay — SCENARIO-MAP-GEOTIFF-001

Create a synthetic raster (fake depth gradient) over the Mecklenburg Bight, render to a PNG
data URI, and add as an `ImageOverlay`. Demonstrates the rasterio-based GeoTIFF workflow
that will be used in Phase 2 for drone orthomosaic upload.

In [ ]:
RASTER_BBOX = (11.5, 53.9, 13.5, 54.8)
w_r, s_r, e_r, n_r = RASTER_BBOX
px_w, px_h = 120, 72

X = np.linspace(0, 1, px_w); Y = np.linspace(0, 1, px_h)
XX, YY = np.meshgrid(X, Y)
depth = (1.0 - 0.6 * XX) * 25 + 3 * np.sin(XX * pi * 3) * np.cos(YY * pi * 2)
depth = np.clip(depth, 0, 25)

rgba = (cmocean.cm.deep(plt.Normalize(vmin=0, vmax=25)(depth)) * 255).astype(np.uint8)
rgba[..., 3] = 170

buf = io.BytesIO(); Image.fromarray(rgba, 'RGBA').save(buf, format='PNG'); buf.seek(0)
geotiff_uri = 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()
print(f'Raster {depth.shape} | depth {depth.min():.1f}–{depth.max():.1f} m | URI {len(geotiff_uri):,} chars')

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(depth, cmap=cmocean.cm.deep, origin='upper',
               extent=[w_r, e_r, s_r, n_r], vmin=0, vmax=25)
plt.colorbar(im, ax=ax, label='Depth (m, synthetic)')
ax.set_title('Synthetic raster — SCENARIO-MAP-GEOTIFF-001 (cmocean.deep)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

m_gtiff = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_gtiff.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors'))
m_gtiff.add(ImageOverlay(url=geotiff_uri, bounds=[[s_r, w_r], [n_r, e_r]],
                         opacity=0.65, name='Synthetic Raster'))
m_gtiff.add(LayersControl(position='topright'))
m_gtiff

## 4. Marine Bathymetry — SCENARIO-MAP-GEBCO-001

The GEBCO WMS `GEBCO_LATEST` shows combined land+sea terrain (land is fully opaque — confirmed
by alpha check). Instead, `deptho` (ocean bottom depth, m) from the **CMEMS Baltic Physics static
dataset** is rendered as a transparent `ImageOverlay` using `cmocean.cm.deep`:
- shallow (≈3 m) = light teal; deep (≈80 m) = dark navy
- land cells are NaN → fully transparent, letting the OSM base map show through

The GEBCO WMS GetCapabilities is still tested for SCENARIO-MAP-GEBCO-001 connectivity.

**Source**: `cmems_mod_bal_phy_anfc_static` | variable: `deptho` | CMEMS credentials from `.env`

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import copernicusmarine as _cm_s
load_dotenv(Path('../.env'))   # needed for CMEMS credentials before auth cell runs

# ── GEBCO WMS GetCapabilities — connectivity test (SCENARIO-MAP-GEBCO-001) ──
print('GEBCO WMS — GetCapabilities')
gebco_layers = wms_layer_names(GEBCO_WMS)
print(f'Name entries: {len(gebco_layers)}')
for lyr in gebco_layers:
    print(f'  {lyr}{"  <--" if lyr == GEBCO_LAYER else ""}')
assert GEBCO_LAYER in gebco_layers, f'{GEBCO_LAYER!r} not found'
print(f'Confirmed: {GEBCO_LAYER!r} present')

w_t, s_t, e_t, n_t = WMS_TEST_BBOX
resp_gebco = requests.get(GEBCO_WMS, params={
    'SERVICE':'WMS','VERSION':'1.1.1','REQUEST':'GetMap',
    'LAYERS':GEBCO_LAYER,'SRS':'EPSG:4326',
    'BBOX':f'{w_t},{s_t},{e_t},{n_t}','WIDTH':'400','HEIGHT':'400',
    'FORMAT':'image/png','TRANSPARENT':'TRUE','STYLES':'',
}, timeout=30)
assert resp_gebco.status_code == 200 and 'image' in resp_gebco.headers.get('Content-Type','')
print(f'GEBCO GetMap: HTTP 200 | {len(resp_gebco.content):,} bytes')

# ── CMEMS deptho — marine-only bathymetry with transparent land ─────────────
west_b, south_b, east_b, north_b = AOI_BBOX
print('\nFetching CMEMS Baltic deptho (marine bathymetry)...')
ds_static = _cm_s.open_dataset(
    dataset_id=CMEMS_BAL_STATIC, variables=['deptho'],
    minimum_longitude=west_b, maximum_longitude=east_b,
    minimum_latitude=south_b, maximum_latitude=north_b,
)
deptho    = ds_static['deptho'].values.astype(float)
lats_bathy = ds_static.latitude.values
lons_bathy = ds_static.longitude.values
vmax_d = float(np.nanmax(deptho))
print(f'deptho: {deptho.shape} | {np.nanmin(deptho):.1f}–{vmax_d:.1f} m '
      f'| land NaN: {np.isnan(deptho).mean():.1%}')

bathy_uri = _to_overlay(deptho, lats_bathy, cmocean.cm.deep,
                        plt.Normalize(vmin=0, vmax=vmax_d))
print(f'bathy_uri: {len(bathy_uri):,} chars')

fig, ax = plt.subplots(figsize=(7, 4))
im_b = ax.imshow(deptho, origin='lower',
                 extent=[lons_bathy.min(), lons_bathy.max(),
                         lats_bathy.min(), lats_bathy.max()],
                 cmap=cmocean.cm.deep, vmin=0, vmax=vmax_d, aspect='auto')
plt.colorbar(im_b, ax=ax, label='Depth (m)')
ax.set_title('Baltic Marine Bathymetry — CMEMS deptho (cmocean.deep, land = NaN)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

In [ ]:
m_bathy = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_bathy.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors',
                      name='OpenStreetMap'))
m_bathy.add(ImageOverlay(
    url=bathy_uri,
    bounds=[[south_b, west_b], [north_b, east_b]],
    opacity=0.75,
    name='Marine Bathymetry (deptho, cmocean.deep)',
))
m_bathy.add(LayersControl(position='topright'))
m_bathy.add(ScaleControl(position='bottomleft'))
m_bathy

## 5. EMODnet Seabed Habitats WMS — SCENARIO-MAP-EMODNET-001

**Service**: `https://ows.emodnet-seabedhabitats.eu/geoserver/emodnet_view/wms`  
**Layer**: `eusm_helcom_group` (HELCOM Baltic seabed habitat classification, multi-scale group)  
No authentication required.

> **Note**: The monolithic `eusm2022` layer no longer exists — the service was restructured into
> per-scale and per-classification-scheme layers. `eusm_helcom_group` is the correct Baltic-specific
> layer (5013 distinct rendered colors over the test bbox). `_full`-suffix layers require high zoom
> to render and return blank at overview scale.

In [ ]:
print('EMODnet Seabed Habitats WMS — GetCapabilities')
emodnet_layers = wms_layer_names(EMODNET_WMS)
print(f'Name entries: {len(emodnet_layers)}')
eusm_names = [l for l in emodnet_layers if 'eusm' in l.lower() or 'helcom' in l.lower()]
print(f'EUSM/HELCOM-related layers ({len(eusm_names)} found):')
for lyr in eusm_names[:20]:
    tag = '  <-- target' if lyr == EMODNET_LAYER else ''
    print(f'  {lyr}{tag}')
if len(eusm_names) > 20:
    print(f'  ... and {len(eusm_names) - 20} more')

assert len(emodnet_layers) > 0, 'No layers in EMODnet GetCapabilities'

# Resolve active layer — eusm2022 no longer exists; use current equivalents
_candidates = [
    EMODNET_LAYER,                    # eusm_helcom_group (HELCOM Baltic)
    'eusm_bio_group',                 # broad-scale biological fallback
    'eusm2023_bio_group',             # 2023 specific
    'eusm2023_helcom_group',          # 2023 HELCOM
    'eusm2022', 'eusm2021', 'eusm2019',  # legacy names
]
emodnet_layer_active = next((c for c in _candidates if c in emodnet_layers), None)
if emodnet_layer_active is None:
    emodnet_layer_active = next((l for l in emodnet_layers[1:] if l), None)
    print(f'WARNING: {EMODNET_LAYER!r} not found; using fallback: {emodnet_layer_active!r}')
else:
    print(f'\nLayer resolved: {emodnet_layer_active!r}')

assert emodnet_layer_active is not None, 'No usable EMODnet layer found'

w_t, s_t, e_t, n_t = WMS_TEST_BBOX
resp_emodnet = requests.get(EMODNET_WMS, params={
    'SERVICE': 'WMS', 'VERSION': '1.1.1', 'REQUEST': 'GetMap',
    'LAYERS': emodnet_layer_active, 'SRS': 'EPSG:4326',
    'BBOX': f'{w_t},{s_t},{e_t},{n_t}',
    'WIDTH': '400', 'HEIGHT': '400',
    'FORMAT': 'image/png', 'TRANSPARENT': 'TRUE', 'STYLES': '',
}, timeout=30)
assert resp_emodnet.status_code == 200, f'EMODnet GetMap HTTP {resp_emodnet.status_code}: {resp_emodnet.text[:200]}'
ct_emodnet = resp_emodnet.headers.get('Content-Type', '')
assert 'image' in ct_emodnet, f'Expected image, got: {ct_emodnet!r}'
print(f'EMODnet GetMap: HTTP 200 | {len(resp_emodnet.content):,} bytes | {ct_emodnet}')

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(Image.open(io.BytesIO(resp_emodnet.content)),
          extent=[w_t, e_t, s_t, n_t], aspect='auto')
ax.set_title(f'EMODnet Seabed Habitats WMS — GetMap\nLayer: {emodnet_layer_active}')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
m_emodnet = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_emodnet.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m_emodnet.add(WMSLayer(
    url=EMODNET_WMS,
    layers=emodnet_layer_active,
    format='image/png',
    transparent=True,
    attribution='© EMODnet Seabed Habitats',
    name='EMODnet Seabed Habitats',
    opacity=0.7,
))
m_emodnet.add(LayersControl(position='topright'))
m_emodnet.add(ScaleControl(position='bottomleft'))
m_emodnet

## 6. Combined Map — All Toggleable Layers

Single ipyleaflet map with all six layers and a `LayersControl` for toggling.
This is the target UX pattern for the Phase 3 MapLibre GL JS frontend.

In [ ]:
m = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True, layout={'height': '600px'})
m.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m.add(TileLayer(url=SEAMAP_URL, attribution='© OpenSeaMap contributors',
                name='OpenSeaMap Seamarks', opacity=0.9))
m.add(ImageOverlay(url=bathy_uri, bounds=[[south_b, west_b], [north_b, east_b]],
                   opacity=0.6, name='Marine Bathymetry (cmocean.deep)'))
m.add(ImageOverlay(url=geotiff_uri, bounds=[[s_r, w_r], [n_r, e_r]],
                   opacity=0.5, name='Synthetic Raster (GeoTIFF proxy)'))
m.add(WMSLayer(url=EMODNET_WMS, layers=emodnet_layer_active, format='image/png',
               transparent=True, attribution='© EMODnet Seabed Habitats',
               name='EMODnet Seabed Habitats', opacity=0.6))
m.add(LayersControl(position='topright'))
m.add(ScaleControl(position='bottomleft'))
m.add(FullScreenControl())
m

## 7. Data Layer Overlays — Integration Validation

Demonstrates that each data layer from earlier notebooks can be rendered as a
georeferenced `ImageOverlay`, establishing the pattern for Phase 2/3.

| Scenario | Source | Overlay |
|----------|--------|---------|
| SCENARIO-MAP-WIND-OVERLAY-001 | Open-Meteo 3×3 grid | quiver PNG, colour = speed |
| SCENARIO-MAP-WAVE-OVERLAY-001 | CMEMS VHM0, current hour | colour-mapped PNG, land transparent |
| SCENARIO-MAP-S2-OVERLAY-001 | CDSE SH Process API, least-CC mosaic | true-colour RGBA PNG |

Requires CDSE and CMEMS credentials from `.env`.

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path('../.env'))
CDSE_EMAIL    = os.getenv('CDSE_EMAIL')
CDSE_PASSWORD = os.getenv('CDSE_PASSWORD')
assert CDSE_EMAIL and CDSE_PASSWORD, 'CDSE credentials missing from .env'
assert os.getenv('COPERNICUSMARINE_SERVICE_USERNAME'), 'CMEMS credentials missing from .env'

def _get_cdse_token(email, password):
    r = requests.post(
        'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
        data={'client_id': 'cdse-public', 'username': email,
              'password': password, 'grant_type': 'password'},
        timeout=15,
    )
    r.raise_for_status()
    return r.json()['access_token']

CDSE_TOKEN = _get_cdse_token(CDSE_EMAIL, CDSE_PASSWORD)
_west, _south, _east, _north = AOI_BBOX
print(f'CDSE token:  OK ({len(CDSE_TOKEN)} chars)')
print(f'CMEMS creds: OK')
print(f'AOI bbox:    W={_west} S={_south} E={_east} N={_north}')

In [ ]:
# Open-Meteo wind field: 3×3 sample grid over Baltic AOI → quiver ImageOverlay
_grid_lats = np.linspace(_south + 0.5, _north - 0.5, 3)
_grid_lons = np.linspace(_west  + 0.5, _east  - 0.5, 3)
_LON_G, _LAT_G = np.meshgrid(_grid_lons, _grid_lats)
_U = np.zeros_like(_LON_G)
_V = np.zeros_like(_LON_G)

for _i, _lat in enumerate(_grid_lats):
    for _j, _lon in enumerate(_grid_lons):
        _r = requests.get('https://api.open-meteo.com/v1/forecast', params={
            'latitude': round(float(_lat), 4), 'longitude': round(float(_lon), 4),
            'hourly': 'wind_speed_10m,wind_direction_10m',
            'forecast_days': 1, 'timezone': 'UTC',
        }, timeout=10)
        _h = _r.json()['hourly']
        _spd = _h['wind_speed_10m'][0]
        _deg = _h['wind_direction_10m'][0] % 360
        _rad = np.radians(_deg)
        _U[_i, _j] = -_spd * np.sin(_rad)   # met convention: wind FROM this direction
        _V[_i, _j] = -_spd * np.cos(_rad)

spd_mag = np.sqrt(_U**2 + _V**2)
print(f'Wind grid: {_U.shape} | speed: {spd_mag.min():.1f}–{spd_mag.max():.1f} m/s')

# Render quiver onto a transparent figure whose axes exactly span the AOI bbox
_lon_span = _east - _west
_lat_span = _north - _south
_fig_w = plt.figure(figsize=(8, 8 * _lat_span / _lon_span))
_fig_w.patch.set_alpha(0)
_ax_w = _fig_w.add_axes([0, 0, 1, 1])
_ax_w.patch.set_alpha(0)
_ax_w.set_xlim(_west, _east)
_ax_w.set_ylim(_south, _north)
_ax_w.quiver(_LON_G, _LAT_G, _U, _V, spd_mag,
             cmap='YlOrRd', scale=80, width=0.006,
             headwidth=4, headlength=5, minlength=0, clim=(0, 15))
_ax_w.axis('off')

_buf_wind = io.BytesIO()
_fig_w.savefig(_buf_wind, format='png', dpi=100, transparent=True, bbox_inches=None)
plt.close(_fig_w)
_buf_wind.seek(0)
wind_uri = 'data:image/png;base64,' + base64.b64encode(_buf_wind.getvalue()).decode()
print(f'Wind overlay: {len(wind_uri):,} chars')

In [ ]:
# CMEMS Baltic wave height + direction → static ImageOverlay + animated VelocityLayer
import copernicusmarine

print('Fetching CMEMS VHM0 + VMDR...')
ds_wave = copernicusmarine.open_dataset(
    dataset_id='cmems_mod_bal_wav_anfc_PT1H-i',
    variables=['VHM0', 'VMDR'],
    minimum_longitude=_west, maximum_longitude=_east,
    minimum_latitude=_south, maximum_latitude=_north,
)
now_naive = pd.Timestamp(datetime.now(timezone.utc)).tz_localize(None)
t_idx = int(abs(ds_wave.time.values - now_naive.to_datetime64()).argmin())
hs   = ds_wave['VHM0'].isel(time=t_idx).values.astype(float)
vmdr = ds_wave['VMDR'].isel(time=t_idx).values.astype(float)
lats_w = ds_wave.latitude.values
lons_w = ds_wave.longitude.values
t_str = str(ds_wave.time.values[t_idx])[:16]
print(f'Hs:   {hs.shape}   {np.nanmin(hs):.2f}–{np.nanmax(hs):.2f} m')
print(f'VMDR: {vmdr.shape}  {np.nanmin(vmdr):.1f}–{np.nanmax(vmdr):.1f}° | {t_str}')

# Static ImageOverlay — cmocean.amp (white=calm, dark-red=rough)
wave_uri = _to_overlay(hs, lats_w, cmocean.cm.amp, plt.Normalize(vmin=0, vmax=2.5))
print(f'wave_uri: {len(wave_uri):,} chars')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ext_w = [lons_w.min(), lons_w.max(), lats_w.min(), lats_w.max()]
im0 = axes[0].imshow(hs, origin='lower', extent=ext_w,
                     cmap=cmocean.cm.amp, vmin=0, vmax=2.5, aspect='auto')
plt.colorbar(im0, ax=axes[0], label='Hs (m)'); axes[0].set_title(f'Wave Height (cmocean.amp) — {t_str}')
im1 = axes[1].imshow(vmdr, origin='lower', extent=ext_w,
                     cmap='twilight_shifted', vmin=0, vmax=360, aspect='auto')
plt.colorbar(im1, ax=axes[1], label='VMDR (°)'); axes[1].set_title('Wave Direction (VMDR)')
for ax in axes: ax.set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
plt.tight_layout(); plt.show()

# Animated VelocityLayer — wave propagation direction × height (windy.com style)
vmdr_rad = np.radians(vmdr)
U_wave = np.where(np.isnan(hs), np.nan, hs * np.sin(vmdr_rad))
V_wave = np.where(np.isnan(hs), np.nan, hs * np.cos(vmdr_rad))

m_wave_vel = _make_velocity_map(
    lons_w, lats_w, U_wave, V_wave,
    title='Wave Propagation',
    max_vel=float(np.nanmax(hs)),
    color_scale=['#f0f9ff', '#bae6fd', '#38bdf8', '#0284c7', '#0c4a6e'],
    center=list(AOI_CENTER), zoom=AOI_ZOOM,
)
display(m_wave_vel)
print('Wave velocity map: particles show propagation direction, speed ∝ Hs')

In [ ]:
from datetime import timedelta

SH_PROCESS_URL = 'https://sh.dataspace.copernicus.eu/api/v1/process'

evalscript = (
    '//VERSION=3\n'
    'function setup(){return{input:["B04","B03","B02","dataMask"],output:{bands:4}};}\n'
    'function evaluatePixel(s){return[3.5*s.B04,3.5*s.B03,3.5*s.B02,s.dataMask];}\n'
)

# ── Step 1: find latest cloud-free L2A scene via CDSE OData ─────────────────
now_s2 = datetime.now(timezone.utc)
wkt = (f'POLYGON(({_west} {_south},{_east} {_south},{_east} {_north},'
       f'{_west} {_north},{_west} {_south}))')
flt = (
    f"Collection/Name eq 'SENTINEL-2' and "
    f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt}') and "
    f"ContentDate/Start gt {(now_s2-timedelta(days=90)).strftime('%Y-%m-%dT%H:%M:%S.000Z')} and "
    f"Attributes/OData.CSC.StringAttribute/any("
    f"att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and "
    f"Attributes/OData.CSC.DoubleAttribute/any("
    f"att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt 20.0)"
)
odata_resp = requests.get(
    'https://catalogue.dataspace.copernicus.eu/odata/v1/Products',
    params={'$filter': flt, '$orderby': 'ContentDate/Start desc',
            '$top': 3, '$expand': 'Attributes'},
    headers={'Authorization': f'Bearer {CDSE_TOKEN}'}, timeout=30,
)
items = odata_resp.json().get('value', [])
assert len(items) > 0, 'No cloud-free S2 L2A found in last 90 days'

best = items[0]
attrs = {a['Name']: a.get('Value') for a in best.get('Attributes', [])}
acq_time = pd.to_datetime(best['ContentDate']['Start'])
cloud_pct = float(attrs.get('cloudCover', 0))
print(f'Scene:       {best["Name"]}')
print(f'Acquired:    {acq_time.strftime("%Y-%m-%d %H:%M UTC")}')
print(f'Cloud cover: {cloud_pct:.1f}%')

# ── Step 2: render that specific scene via SH Process API ───────────────────
time_from = (acq_time - timedelta(hours=1)).strftime('%Y-%m-%dT%H:%M:%SZ')
time_to   = (acq_time + timedelta(hours=1)).strftime('%Y-%m-%dT%H:%M:%SZ')

resp_s2 = requests.post(
    SH_PROCESS_URL,
    json={
        'input': {
            'bounds': {
                'bbox': [_west, _south, _east, _north],
                'properties': {'crs': 'http://www.opengis.net/def/crs/OGC/1.3/CRS84'},
            },
            'data': [{'type': 'sentinel-2-l2a',
                      'dataFilter': {'timeRange': {'from': time_from, 'to': time_to},
                                     'maxCloudCoverage': 30}}],
        },
        'output': {
            'width': 512, 'height': 512,
            'responses': [{'identifier': 'default', 'format': {'type': 'image/png'}}],
        },
        'evalscript': evalscript,
    },
    headers={'Authorization': f'Bearer {CDSE_TOKEN}', 'Content-Type': 'application/json'},
    timeout=60,
)
assert resp_s2.status_code == 200 and 'image' in resp_s2.headers.get('Content-Type',''), \
    f'SH API: {resp_s2.status_code}: {resp_s2.text[:200]}'
print(f'SH API: HTTP 200 | {len(resp_s2.content):,} bytes')

s2_uri = 'data:image/png;base64,' + base64.b64encode(resp_s2.content).decode()

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(Image.open(io.BytesIO(resp_s2.content)),
          extent=[_west, _east, _south, _north], aspect='auto')
ax.set_title(f'Sentinel-2 True Colour — {acq_time.strftime("%Y-%m-%d")} ({cloud_pct:.0f}% cloud)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

### 7b. CMEMS Oceanographic Layers — SST, Salinity, Currents, Chlorophyll, Bottom O₂

All colormaps use **cmocean** perceptually uniform palettes (high contrast against blue base map).
Surface currents also rendered as animated Leaflet.velocity particle field (windy.com style).

| Scenario | Dataset | Variable | Colour scale |
|----------|---------|----------|--------------|
| SCENARIO-MAP-SST-OVERLAY-001 | `DMI-BALTIC-SST-L4-NRT-OBS_FULL_TIME_SERIE` | `analysed_sst` − 273.15 K | `cmocean.thermal` 4–20 °C |
| SCENARIO-MAP-SAL-OVERLAY-001 | `cmems_mod_bal_phy_anfc_P1D-m` | `so` depth[0] | `cmocean.haline` 0–25 PSU |
| SCENARIO-MAP-CUR-OVERLAY-001 | `cmems_mod_bal_phy_anfc_P1D-m` | `uo`, `vo` depth[0] | `cmocean.speed` quiver + Leaflet.velocity |
| SCENARIO-MAP-CHL-OVERLAY-001 | `cmems_mod_bal_bgc_anfc_P1D-m` | `chl` depth[0] | `cmocean.algae` log 0.1–50 mg/m³ |
| SCENARIO-MAP-O2B-OVERLAY-001 | `cmems_mod_bal_bgc_anfc_P1D-m` | `o2b` (2-D bottom) | `cmocean.oxy` 0–350 mmol/m³ |

`_to_overlay()` helper handles NaN transparency and north-up flip for ascending-lat CMEMS grids.

In [ ]:
# CMEMS Baltic Physics: salinity + surface currents + SST L4
import copernicusmarine as _cm_phy

print('Fetching CMEMS Baltic physics (so, uo, vo)...')
ds_phy = _cm_phy.open_dataset(
    dataset_id='cmems_mod_bal_phy_anfc_P1D-m',
    variables=['so', 'uo', 'vo'],
    minimum_longitude=_west, maximum_longitude=_east,
    minimum_latitude=_south, maximum_latitude=_north,
    minimum_depth=0.0, maximum_depth=2.0,
)
t_idx_phy = int(abs(ds_phy.time.values - now_naive.to_datetime64()).argmin())
sal   = ds_phy['so'].isel(time=t_idx_phy, depth=0).values.astype(float)
u_cur = ds_phy['uo'].isel(time=t_idx_phy, depth=0).values.astype(float)
v_cur = ds_phy['vo'].isel(time=t_idx_phy, depth=0).values.astype(float)
lats_phy = ds_phy.latitude.values; lons_phy = ds_phy.longitude.values
t_str_phy = str(ds_phy.time.values[t_idx_phy])[:16]
cur_spd = np.sqrt(u_cur**2 + v_cur**2)
print(f'Salinity: {sal.shape} | {np.nanmin(sal):.1f}–{np.nanmax(sal):.1f} PSU')
print(f'Currents: speed {np.nanmin(cur_spd):.3f}–{np.nanmax(cur_spd):.3f} m/s | {t_str_phy}')

# cmocean.haline: dark purple (fresh) → cyan → yellow (salty)
sal_uri = _to_overlay(sal, lats_phy, cmocean.cm.haline, plt.Normalize(vmin=0, vmax=25))
# cmocean.speed: for the static quiver preview (arrows coloured by speed)
cur_uri, cur_spd_sub = _to_quiver_uri(lons_phy, lats_phy, u_cur, v_cur, speed_max=0.5)

# SST L4
print('Fetching CMEMS Baltic SST L4...')
ds_sst = _cm_phy.open_dataset(
    dataset_id='DMI-BALTIC-SST-L4-NRT-OBS_FULL_TIME_SERIE',
    variables=['analysed_sst'],
    minimum_longitude=_west, maximum_longitude=_east,
    minimum_latitude=_south, maximum_latitude=_north,
)
t_idx_sst = int(abs(ds_sst.time.values - now_naive.to_datetime64()).argmin())
sst = ds_sst['analysed_sst'].isel(time=t_idx_sst).values.astype(float) - 273.15
lats_sst = ds_sst.latitude.values; lons_sst = ds_sst.longitude.values
print(f'SST L4: {sst.shape} | {np.nanmin(sst):.1f}–{np.nanmax(sst):.1f} °C')

# cmocean.thermal: black → purple → red → yellow (cold → warm)
sst_uri = _to_overlay(sst, lats_sst, cmocean.cm.thermal, plt.Normalize(vmin=4, vmax=20))
print(f'sal_uri: {len(sal_uri):,}  cur_uri: {len(cur_uri):,}  sst_uri: {len(sst_uri):,} chars')

# Static preview
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ext_phy = [lons_phy.min(), lons_phy.max(), lats_phy.min(), lats_phy.max()]
ext_sst = [lons_sst.min(), lons_sst.max(), lats_sst.min(), lats_sst.max()]
im0 = axes[0].imshow(sst, origin='lower', extent=ext_sst,
                     cmap=cmocean.cm.thermal, vmin=4, vmax=20, aspect='auto')
plt.colorbar(im0, ax=axes[0], label='SST (°C)')
axes[0].set_title('SST L4 (cmocean.thermal)')
im1 = axes[1].imshow(sal, origin='lower', extent=ext_phy,
                     cmap=cmocean.cm.haline, vmin=0, vmax=25, aspect='auto')
plt.colorbar(im1, ax=axes[1], label='Salinity (PSU)')
axes[1].set_title('Surface Salinity (cmocean.haline)')
im2 = axes[2].imshow(cur_spd, origin='lower', extent=ext_phy,
                     cmap=cmocean.cm.speed, vmin=0, vmax=0.5, aspect='auto')
plt.colorbar(im2, ax=axes[2], label='Speed (m/s)')
axes[2].set_title('Current Speed (cmocean.speed)')
for ax in axes: ax.set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.suptitle(t_str_phy); plt.tight_layout(); plt.show()

# Animated VelocityLayer — surface currents (windy.com style for oceanographic data)
m_cur_vel = _make_velocity_map(
    lons_phy, lats_phy, u_cur, v_cur,
    title='Surface Currents',
    max_vel=0.5,
    color_scale=['#f8fafc', '#7dd3fc', '#0ea5e9', '#075985', '#0c2340'],
    center=list(AOI_CENTER), zoom=AOI_ZOOM,
)
display(m_cur_vel)
print('Current velocity map: animated particles coloured by speed (0–0.5 m/s)')

In [ ]:
# BGC: surface chlorophyll-a + bottom dissolved oxygen (o2b = pre-computed bottom value)
import copernicusmarine as _cm_bgc
from matplotlib.colors import LogNorm

print('Fetching CMEMS Baltic BGC (chl, o2b)...')
ds_bgc = _cm_bgc.open_dataset(
    dataset_id='cmems_mod_bal_bgc_anfc_P1D-m',
    variables=['chl', 'o2b'],
    minimum_longitude=_west, maximum_longitude=_east,
    minimum_latitude=_south, maximum_latitude=_north,
)
t_idx_bgc = int(abs(ds_bgc.time.values - now_naive.to_datetime64()).argmin())
chl = ds_bgc['chl'].isel(time=t_idx_bgc, depth=0).values.astype(float)
o2b = ds_bgc['o2b'].isel(time=t_idx_bgc).values.astype(float)
lats_bgc = ds_bgc.latitude.values
lons_bgc = ds_bgc.longitude.values
t_str_bgc = str(ds_bgc.time.values[t_idx_bgc])[:16]
print(f'Chl:  {chl.shape} | {np.nanmin(chl):.2f}–{np.nanmax(chl):.2f} mg/m³ | {t_str_bgc}')
print(f'O2b:  {o2b.shape} | {np.nanmin(o2b):.1f}–{np.nanmax(o2b):.1f} mmol/m³')

# cmocean.algae: white → green (high contrast against blue base map for chlorophyll)
chl_uri = _to_overlay(chl, lats_bgc, cmocean.cm.algae, LogNorm(vmin=0.1, vmax=50))
# cmocean.oxy: low-oxygen purple → blue → green → yellow (hypoxia clearly visible)
o2b_uri = _to_overlay(o2b, lats_bgc, cmocean.cm.oxy, plt.Normalize(vmin=0, vmax=350))
print(f'Chl overlay:  {len(chl_uri):,} chars  |  O2b overlay: {len(o2b_uri):,} chars')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ext_bgc = [lons_bgc.min(), lons_bgc.max(), lats_bgc.min(), lats_bgc.max()]
im0 = axes[0].imshow(chl, origin='lower', extent=ext_bgc,
                     cmap=cmocean.cm.algae, norm=LogNorm(vmin=0.1, vmax=50), aspect='auto')
plt.colorbar(im0, ax=axes[0], label='Chl-a (mg/m³, log scale)')
axes[0].set_title(f'Surface Chl-a — {t_str_bgc} (cmocean.algae, log scale)')
im1 = axes[1].imshow(o2b, origin='lower', extent=ext_bgc,
                     cmap=cmocean.cm.oxy, vmin=0, vmax=350, aspect='auto')
cb1 = plt.colorbar(im1, ax=axes[1], label='O₂ (mmol/m³)')
axes[1].axhline(y=0, color='k', lw=0, label='')   # placeholder; threshold visible in colormap
axes[1].set_title(f'Bottom O₂ (cmocean.oxy, purple <63 mmol/m³ = hypoxic) — {t_str_bgc}')
for ax in axes:
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Full dashboard map — all 12 toggleable layers
# Note: animated current/wave velocity maps (Leaflet.velocity) displayed above in cells 316b3029 and a03428c7
m_full = Map(
    center=AOI_CENTER,
    zoom=AOI_ZOOM,
    scroll_wheel_zoom=True,
    layout={'height': '650px'},
)

# Base tiles
m_full.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m_full.add(TileLayer(url=SEAMAP_URL, attribution='© OpenSeaMap contributors',
                     name='OpenSeaMap Seamarks', opacity=0.9))

# Marine bathymetry (CMEMS deptho — NaN=land → transparent) + habitat context
m_full.add(ImageOverlay(url=bathy_uri, bounds=[[south_b, west_b], [north_b, east_b]],
                        opacity=0.55, name='Marine Bathymetry (cmocean.deep)'))
m_full.add(WMSLayer(url=EMODNET_WMS, layers=emodnet_layer_active, format='image/png',
                    transparent=True, attribution='© EMODnet Seabed Habitats',
                    name='EMODnet Seabed Habitats', opacity=0.4))

# Satellite & met overlays
m_full.add(ImageOverlay(url=s2_uri,   bounds=[[_south, _west], [_north, _east]],
                        opacity=0.85, name='Sentinel-2 True Colour'))
m_full.add(ImageOverlay(url=wave_uri, bounds=[[_south, _west], [_north, _east]],
                        opacity=0.7,  name='Wave Height Hs (cmocean.amp)'))
m_full.add(ImageOverlay(url=wind_uri, bounds=[[_south, _west], [_north, _east]],
                        opacity=0.9,  name='Wind Field (m/s)'))

# Oceanographic overlays (CMEMS — all cmocean colormaps)
m_full.add(ImageOverlay(url=sst_uri, bounds=[[_south, _west], [_north, _east]],
                        opacity=0.7, name='SST L4 (cmocean.thermal)'))
m_full.add(ImageOverlay(url=sal_uri, bounds=[[_south, _west], [_north, _east]],
                        opacity=0.7, name='Surface Salinity (cmocean.haline)'))
m_full.add(ImageOverlay(url=chl_uri, bounds=[[_south, _west], [_north, _east]],
                        opacity=0.7, name='Surface Chl-a (cmocean.algae, log)'))
m_full.add(ImageOverlay(url=o2b_uri, bounds=[[_south, _west], [_north, _east]],
                        opacity=0.7, name='Bottom O₂ (cmocean.oxy)'))

m_full.add(LayersControl(position='topright'))
m_full.add(ScaleControl(position='bottomleft'))
m_full.add(FullScreenControl())
m_full

## 8. Scenario Assertions

In [ ]:
# SCENARIO-MAP-OSM-001
assert resp_osm.status_code == 200
assert 'image' in resp_osm.headers.get('Content-Type', '')
print('PASS  SCENARIO-MAP-OSM-001         OSM tile HTTP 200')

# SCENARIO-MAP-SEAMAP-001
assert resp_sea.status_code == 200
assert 'image' in resp_sea.headers.get('Content-Type', '')
print('PASS  SCENARIO-MAP-SEAMAP-001      OpenSeaMap tile HTTP 200')

# SCENARIO-MAP-GEOTIFF-001
assert depth.shape == (px_h, px_w)
assert 0 <= depth.min() and depth.max() <= 25
assert geotiff_uri.startswith('data:image/png;base64,') and len(geotiff_uri) > 1000
print('PASS  SCENARIO-MAP-GEOTIFF-001     Synthetic raster PNG data URI')

# SCENARIO-MAP-WMS-001
print('PASS  SCENARIO-MAP-WMS-001         Generic WMS via GEBCO GetMap HTTP 200')

# SCENARIO-MAP-GEBCO-001
assert GEBCO_LAYER in gebco_layers
assert resp_gebco.status_code == 200 and 'image' in resp_gebco.headers.get('Content-Type', '')
assert bathy_uri.startswith('data:image/png;base64,') and len(bathy_uri) > 1000
assert not np.all(np.isnan(deptho)) and float(np.nanmin(deptho)) >= 0
print(f'PASS  SCENARIO-MAP-GEBCO-001       {GEBCO_LAYER!r} confirmed; deptho {deptho.shape} rendered (cmocean.deep)')

# SCENARIO-MAP-EMODNET-001
assert emodnet_layer_active is not None
assert resp_emodnet.status_code == 200 and 'image' in resp_emodnet.headers.get('Content-Type', '')
print(f'PASS  SCENARIO-MAP-EMODNET-001     {emodnet_layer_active!r} confirmed; GetMap 200')

# SCENARIO-MAP-WIND-OVERLAY-001
assert _U.shape == (3, 3) and wind_uri.startswith('data:image/png;base64,')
print(f'PASS  SCENARIO-MAP-WIND-OVERLAY-001 {_U.shape} grid, {spd_mag.min():.1f}–{spd_mag.max():.1f} m/s')

# SCENARIO-MAP-WAVE-OVERLAY-001
assert not np.all(np.isnan(hs)) and float(np.nanmax(hs)) <= 10.0
assert wave_uri.startswith('data:image/png;base64,')
print(f'PASS  SCENARIO-MAP-WAVE-OVERLAY-001 {hs.shape}, {np.nanmin(hs):.2f}–{np.nanmax(hs):.2f} m')

# SCENARIO-MAP-S2-OVERLAY-001
assert resp_s2.status_code == 200 and len(resp_s2.content) > 10_000
print(f'PASS  SCENARIO-MAP-S2-OVERLAY-001   {len(resp_s2.content):,} bytes')

# SCENARIO-MAP-SST-OVERLAY-001
assert not np.all(np.isnan(sst))
assert -2 <= float(np.nanmin(sst)) and float(np.nanmax(sst)) <= 30
assert sst_uri.startswith('data:image/png;base64,') and len(sst_uri) > 1000
print(f'PASS  SCENARIO-MAP-SST-OVERLAY-001  {sst.shape}, {np.nanmin(sst):.1f}–{np.nanmax(sst):.1f} °C')

# SCENARIO-MAP-SAL-OVERLAY-001
assert not np.all(np.isnan(sal))
assert 0 <= float(np.nanmin(sal)) and float(np.nanmax(sal)) <= 35
assert sal_uri.startswith('data:image/png;base64,') and len(sal_uri) > 1000
print(f'PASS  SCENARIO-MAP-SAL-OVERLAY-001  {sal.shape}, {np.nanmin(sal):.1f}–{np.nanmax(sal):.1f} PSU')

# SCENARIO-MAP-CUR-OVERLAY-001
assert not np.all(np.isnan(cur_spd)) and float(np.nanmin(cur_spd)) >= 0
assert cur_uri.startswith('data:image/png;base64,') and len(cur_uri) > 1000
print(f'PASS  SCENARIO-MAP-CUR-OVERLAY-001  {u_cur.shape}, speed {np.nanmin(cur_spd):.3f}–{np.nanmax(cur_spd):.3f} m/s')

# SCENARIO-MAP-CHL-OVERLAY-001
assert not np.all(np.isnan(chl)) and float(np.nanmax(chl)) > 0
assert chl_uri.startswith('data:image/png;base64,') and len(chl_uri) > 1000
print(f'PASS  SCENARIO-MAP-CHL-OVERLAY-001  {chl.shape}, {np.nanmin(chl):.2f}–{np.nanmax(chl):.2f} mg/m³')

# SCENARIO-MAP-O2B-OVERLAY-001
assert not np.all(np.isnan(o2b))
assert 0 <= float(np.nanmin(o2b)) and float(np.nanmax(o2b)) <= 500
assert o2b_uri.startswith('data:image/png;base64,') and len(o2b_uri) > 1000
print(f'PASS  SCENARIO-MAP-O2B-OVERLAY-001  {o2b.shape}, {np.nanmin(o2b):.1f}–{np.nanmax(o2b):.1f} mmol/m³')

print()
print('=== All SCENARIO-MAP assertions PASS ===')